# Magic Methods (Dunder Methods)

**Magic methods** (also called **dunder methods** because they have `__double_underscores__`) let your custom classes behave like built-in Python types.

They are called automatically by Python in response to operations:
- `+` calls `__add__`
- `print()` calls `__str__`
- `len()` calls `__len__`
- `for x in obj` calls `__iter__`
- `with obj as x` calls `__enter__` and `__exit__`

By the end of this notebook you will be able to:
- Implement the most important magic methods
- Make your classes support arithmetic, comparison, and iteration
- Create callable objects and context managers

## 1. String Representation: `__str__` and `__repr__`

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __str__(self):
        """Human-readable: used by print()"""
        return f'({self.x}, {self.y})'
    
    def __repr__(self):
        """Developer-readable: used in REPL, lists"""
        return f'Point({self.x!r}, {self.y!r})'

p = Point(3, 4)
print(str(p))   # (3, 4)
print(repr(p))  # Point(3, 4)
print(p)        # (3, 4) — uses __str__
print([p])      # [Point(3, 4)] — lists use __repr__

## 2. Arithmetic Operators

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __str__(self):
        return f'Vector({self.x}, {self.y})'
    
    def __add__(self, other):       # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):       # v1 - v2
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):      # v * 3
        return Vector(self.x * scalar, self.y * scalar)
    
    def __rmul__(self, scalar):     # 3 * v  (right multiply)
        return self.__mul__(scalar)
    
    def __neg__(self):              # -v
        return Vector(-self.x, -self.y)
    
    def __abs__(self):              # abs(v) — magnitude
        return (self.x**2 + self.y**2) ** 0.5

v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(v1 + v2)    # Vector(4, 6)
print(v2 - v1)    # Vector(2, 2)
print(v1 * 3)     # Vector(3, 6)
print(3 * v1)     # Vector(3, 6)
print(-v1)        # Vector(-1, -2)
print(abs(v2))    # 5.0

## 3. Comparison Operators

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    
    def __eq__(self, other):   # ==
        return self.celsius == other.celsius
    
    def __lt__(self, other):   # <
        return self.celsius < other.celsius
    
    def __le__(self, other):   # <=
        return self.celsius <= other.celsius
    
    def __gt__(self, other):   # >
        return self.celsius > other.celsius
    
    def __ge__(self, other):   # >=
        return self.celsius >= other.celsius
    
    def __str__(self):
        return f'{self.celsius}°C'

t1 = Temperature(20)
t2 = Temperature(30)
t3 = Temperature(20)

print(t1 == t3)   # True
print(t1 < t2)    # True
print(t2 > t1)    # True

# Now we can sort!
temps = [Temperature(35), Temperature(10), Temperature(22), Temperature(5)]
sorted_temps = sorted(temps)
print([str(t) for t in sorted_temps])

## 4. Container Methods: `__len__`, `__getitem__`, `__contains__`

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self._songs = []
    
    def add(self, song):
        self._songs.append(song)
    
    def __len__(self):          # len(playlist)
        return len(self._songs)
    
    def __getitem__(self, index): # playlist[i] or playlist[1:3]
        return self._songs[index]
    
    def __contains__(self, song): # 'song' in playlist
        return song in self._songs
    
    def __str__(self):
        return f'Playlist "{self.name}" ({len(self)} songs)'

pl = Playlist('Favorites')
for song in ['Bohemian Rhapsody', 'Stairway to Heaven', 'Hotel California', 'Imagine']:
    pl.add(song)

print(pl)
print(f'Length: {len(pl)}')
print(f'First: {pl[0]}')
print(f'Last: {pl[-1]}')
print(f'Slice: {pl[1:3]}')
print(f'Has Imagine: {"Imagine" in pl}')
print(f'Has Yesterday: {"Yesterday" in pl}')

## 5. Iteration: `__iter__` and `__next__`

Make your class work with `for` loops by implementing the **iterator protocol**.

In [ ]:
class CountUp:
    """Iterator that counts from start to stop."""
    
    def __init__(self, start, stop):
        self.start = start
        self.stop = stop
        self.current = start
    
    def __iter__(self):  # called when 'for x in obj' starts
        self.current = self.start
        return self
    
    def __next__(self):  # called for each iteration
        if self.current > self.stop:
            raise StopIteration  # tells for-loop to stop
        value = self.current
        self.current += 1
        return value

counter = CountUp(1, 5)

# Use in a for loop
for n in counter:
    print(n, end=' ')
print()

# Use multiple times
print(list(counter))    # [1, 2, 3, 4, 5]
print(sum(counter))     # 15

## 6. Context Manager: `__enter__` and `__exit__`

Makes your class usable with `with` statements for resource management.

In [ ]:
import time

class Timer:
    """Context manager that times a block of code."""
    
    def __enter__(self):
        self.start = time.time()
        print('Timer started')
        return self  # value bound to 'as' variable
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.time() - self.start
        print(f'Timer stopped: {self.elapsed:.4f}s')
        return False  # False = don't suppress exceptions

with Timer() as t:
    # Code being timed
    total = sum(range(1_000_000))

print(f'Sum: {total}')
print(f'Elapsed: {t.elapsed:.4f}s')

## 7. Callable Objects: `__call__`

Implement `__call__` to make instances callable like functions.

In [ ]:
class Multiplier:
    """A callable that multiplies by a fixed factor."""
    
    def __init__(self, factor):
        self.factor = factor
    
    def __call__(self, value):  # called when instance is called as a function
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15
print(double(triple(4)))  # double(12) = 24

# Works with map()
numbers = [1, 2, 3, 4, 5]
print(list(map(double, numbers)))  # [2, 4, 6, 8, 10]

## 8. `__bool__` and `__hash__`

In [ ]:
class Wallet:
    def __init__(self, balance):
        self.balance = balance
    
    def __bool__(self):   # controls truthiness
        return self.balance > 0
    
    def __str__(self):
        return f'Wallet(£{self.balance:.2f})'

full_wallet = Wallet(50)
empty_wallet = Wallet(0)

print(bool(full_wallet))   # True
print(bool(empty_wallet))  # False

if full_wallet:
    print('Can make a purchase!')

if not empty_wallet:
    print('Wallet is empty — need to top up')

## Common Mistakes

### Mistake 1: `__eq__` Without `__hash__` (Breaking Hashability)

In [ ]:
# If you define __eq__, Python automatically sets __hash__ = None
# This makes your object unhashable (can't be used as dict key or in a set)

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    
    # Need to also define __hash__ if you want it hashable
    def __hash__(self):
        return hash((self.x, self.y))

p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)

print(p1 == p2)  # True
print(p1 == p3)  # False

# Now usable in sets and as dict keys
point_set = {p1, p2, p3}
print(len(point_set))  # 2 — p1 and p2 are equal

## Exercises

### Exercise 1 (Easy): Fraction Class — String Representation
Create a `Fraction(numerator, denominator)` class with `__str__` and `__repr__`.

In [ ]:
# Your solution here


In [ ]:
# Solution
from math import gcd

class Fraction:
    def __init__(self, numerator, denominator):
        if denominator == 0:
            raise ValueError('Denominator cannot be zero')
        common = gcd(abs(numerator), abs(denominator))
        self.num = numerator // common
        self.den = denominator // common
    
    def __str__(self):
        return f'{self.num}/{self.den}'
    
    def __repr__(self):
        return f'Fraction({self.num}, {self.den})'
    
    def __add__(self, other):
        new_num = self.num * other.den + other.num * self.den
        new_den = self.den * other.den
        return Fraction(new_num, new_den)
    
    def __eq__(self, other):
        return self.num == other.num and self.den == other.den

f1 = Fraction(1, 2)
f2 = Fraction(1, 3)
print(f1, '+', f2, '=', f1 + f2)
print(Fraction(2, 4))   # automatically reduced to 1/2

## Mini-Project: Custom Vector Class

A full-featured Vector class supporting arithmetic, comparison, and string representation.

In [ ]:
import math

class Vector:
    def __init__(self, *components):
        self.components = tuple(components)
    
    def __str__(self):
        return f'Vector{self.components}'
    
    def __repr__(self):
        args = ', '.join(str(c) for c in self.components)
        return f'Vector({args})'
    
    def __len__(self):
        return len(self.components)
    
    def __getitem__(self, index):
        return self.components[index]
    
    def __add__(self, other):
        return Vector(*[a + b for a, b in zip(self.components, other.components)])
    
    def __sub__(self, other):
        return Vector(*[a - b for a, b in zip(self.components, other.components)])
    
    def __mul__(self, scalar):
        return Vector(*[c * scalar for c in self.components])
    
    def __rmul__(self, scalar):
        return self.__mul__(scalar)
    
    def __eq__(self, other):
        return self.components == other.components
    
    def __abs__(self):
        return math.sqrt(sum(c**2 for c in self.components))
    
    def __neg__(self):
        return Vector(*[-c for c in self.components])
    
    def dot(self, other):
        return sum(a * b for a, b in zip(self.components, other.components))
    
    def normalize(self):
        magnitude = abs(self)
        if magnitude == 0:
            raise ValueError('Cannot normalize zero vector')
        return Vector(*[c / magnitude for c in self.components])


# Test
v1 = Vector(1, 2, 3)
v2 = Vector(4, 5, 6)

print(f'v1 = {v1}')
print(f'v2 = {v2}')
print(f'v1 + v2 = {v1 + v2}')
print(f'v2 - v1 = {v2 - v1}')
print(f'v1 * 3 = {v1 * 3}')
print(f'|v1| = {abs(v1):.4f}')
print(f'v1 · v2 = {v1.dot(v2)}')
print(f'v1 == v1: {v1 == Vector(1,2,3)}')
n = v1.normalize()
print(f'unit v1 = ({n[0]:.3f}, {n[1]:.3f}, {n[2]:.3f})')